# PROVA NEURAL NETWORK IN SIMBIOSI CON SKLEARN

---

Leggo il nostro dict in `.pkl`

In [1]:
import pickle as pkl

with open('data/ottani_NIR.pkl', 'rb') as f:
    dizionario = pkl.load(f)
    
# dati train
train_ones = dizionario['train']['NIR']['ones']
train_zeros = dizionario['train']['NIR']['zeros']
train_labels = dizionario['train']['labels']

# dati test
test_ones = dizionario['test']['NIR']['ones']
test_zeros = dizionario['test']['NIR']['zeros']
test_labels = dizionario['test']['labels']


In [2]:
import numpy as np

train_x = np.concatenate((train_zeros, train_ones))
test_x = np.concatenate((test_zeros,test_ones))


importo neural network

In [3]:
from ML_app.single_neuron import single_neuron as neural_network

---

## SMOOTHING 

provo a filtrare i dati con Savitzky-Golay

In [4]:
from scipy.signal import savgol_filter

window_size = 11
poly_order = 3

for i in range(train_x.shape[0]):
    train_x[i] = savgol_filter(train_x[i], window_size, poly_order)

for j in range(test_x.shape[0]):
    test_x[j] = savgol_filter(test_x[j], window_size, poly_order)

---

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

26 minuti

In [6]:
rkf = RepeatedStratifiedKFold(n_splits=6, n_repeats=10, random_state=42)

# SCALING
SCALER_OPTIONS = [MinMaxScaler()]
# PCA
N_COMPONENTS_OPTIONS = [2, 5, 7, 19, 37, None]
# ESTIMATOR
LEARNING_RATE_OPTIONS = [0.001, 0.1, 0.5, 0.9, 0.95]
#EPOCHE_OPTIONS = [1_000, 2_000, 10_000, 20_000]
CONVERGENCE_TOL_OPTIONS = [1e-03, 5e-03, 1e-02]
LOSS_FUNCTION_OPTIONS = ['MSE']
ACTIVATION_FUNCTION_OPTIONS = ['sigmoide', 'tanh']

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling
    ("scaling", MinMaxScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=None)),
    
    # Step 3: Classificatore
    ("classify", neural_network(random_state=None, epoche=1_000_000)) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per provare diversi numeri di componenti (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per cambiare parametri del neurone
    #"classify__epoche": EPOCHE_OPTIONS,
    "classify__convergence_tol": CONVERGENCE_TOL_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
    "classify__activation_function": ACTIVATION_FUNCTION_OPTIONS,
    
},
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per saltare la riduzione
    "reduce_dim": ['passthrough'], 
    
    # Per cambiare parametri del neurone
    #"classify__epoche": EPOCHE_OPTIONS,
    "classify__convergence_tol": CONVERGENCE_TOL_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
    "classify__activation_function": ACTIVATION_FUNCTION_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
        'score': 'accuracy',
        'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
)

# 4. Training e Validation (su Segnale B) #
grid.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(test_x, test_labels)
print(f"Risultato sul set indipendente: {accuracy_finale:.4f}")

# Quanto ha impiegato il miglior estimator?
#print('convergenza in:', grid.best_estimator_._convergence_idx)

La miglior configurazione: {'classify__activation_function': 'tanh', 'classify__convergence_tol': 0.005, 'classify__nn_learning_rate': 0.1, 'reduce_dim__n_components': 5, 'scaling': MinMaxScaler()}
Fornisce accuracy in validation: 0.9667
Risultato sul set indipendente: 0.9167


In [7]:
import pandas as pd
# Conversione dei risultati in DataFrame
results_df = pd.DataFrame(grid.cv_results_)
results_df.to_pickle("results/results-nn-savgol-GridSearch.pkl")

In [9]:
import pandas as pd

results_df = pd.DataFrame(grid.cv_results_)
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_scaling',
    'param_reduce_dim__n_components', 
    'param_classify__activation_function',
    'param_classify__convergence_tol',
    #'param_classify__epoche',
    'param_classify__nn_learning_rate', 
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'std_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis = results_df[columns_to_show].sort_values('rank_test_score')
renamed = analysis.rename(columns={
    'param_scaling': 'scaler',
    'param_reduce_dim__n_components': 'pca_n_components', 
    'param_classify__activation_function': 'activation_function',
    'param_classify__convergence_tol': 'convergence_tol',
    #'param_classify__epoche': 'epoche',
    'param_classify__nn_learning_rate': 'learning_rate', 
    'mean_test_score': 'mean_score', 
    'std_test_score': 'std_score', 
    'mean_test_sensitivity': 'mean_sensitivity',
    'std_test_sensitivity': 'std_sensitivity',
    'rank_test_score': 'rank_score',
})

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
renamed.head(20)

Numero totale di configurazioni provate: 210


,scaler,pca_n_components,activation_function,convergence_tol,learning_rate,mean_score,std_score,mean_sensitivity,std_sensitivity,rank_score
127,MinMaxScaler(),5,tanh,0.005,0.100,0.966667,0.078617,0.975000,0.087797,1
97,MinMaxScaler(),5,tanh,0.001,0.100,0.956250,0.109032,0.966667,0.106719,2
157,MinMaxScaler(),5,tanh,0.010,0.100,0.956250,0.096353,0.950000,0.127475,2
15,MinMaxScaler(),19,sigmoide,0.001,0.500,0.950000,0.065352,0.995833,0.032005,4
41,MinMaxScaler(),None,sigmoide,0.005,0.100,0.947917,0.065716,0.995833,0.032005,5
70,MinMaxScaler(),37,sigmoide,0.010,0.100,0.947917,0.065716,0.995833,0.032005,5
69,MinMaxScaler(),19,sigmoide,0.010,0.100,0.947917,0.065716,0.995833,0.032005,5
45,MinMaxScaler(),19,sigmoide,0.005,0.500,0.947917,0.065716,0.995833,0.032005,5
16,MinMaxScaler(),37,sigmoide,0.001,0.500,0.947917,0.065716,0.995833,0.032005,5
39,MinMaxScaler(),19,sigmoide,0.005,0.100,0.947917,0.065716,0.995833,0.032005,5


---

## fine tuning

46 minuti

In [ ]:
rkf = RepeatedStratifiedKFold(n_splits=6, n_repeats=10, random_state=42)

# SCALING
SCALER_OPTIONS = [MinMaxScaler()]
# PCA
N_COMPONENTS_OPTIONS = [35, 37, 40]
# ESTIMATOR
LEARNING_RATE_OPTIONS = [0.001, 0.1, 0.5, 0.9]
EPOCHE_OPTIONS = [20_000, 100_000, 500_000, 1_000_000]
LOSS_FUNCTION_OPTIONS = ['MSE']
ACTIVATION_FUNCTION_OPTIONS = ['sigmoide']

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling
    ("scaling", MinMaxScaler()),       
    
    # Step 2: Riduzione dimensionalità (PCA)
    ("reduce_dim", PCA(random_state=None)),
    
    # Step 3: Classificatore
    ("classify", neural_network(random_state=None)) 
])

# 2. Definizione griglia dei parametri #
param_grid = [
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per provare diversi numeri di componenti (PCA)
    "reduce_dim__n_components": N_COMPONENTS_OPTIONS, 
    
    # Per cambiare parametri del neurone
    "classify__epoche": EPOCHE_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
    "classify__activation_function": ACTIVATION_FUNCTION_OPTIONS,
    
},
{
    # Per provare diversi scaler
    "scaling": SCALER_OPTIONS,
    
    # Per saltare la riduzione
    "reduce_dim": ['passthrough'], 
    
    # Per cambiare parametri del neurone
    "classify__epoche": EPOCHE_OPTIONS,
    "classify__nn_learning_rate": LEARNING_RATE_OPTIONS,
    "classify__activation_function": ACTIVATION_FUNCTION_OPTIONS,
}]

# 3. Configurazione GridSearch #
grid_fine = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf,
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
        'score': 'accuracy',
        'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
)

# 4. Training e Validation (su Segnale B) #
grid_fine.fit(train_x, train_labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid_fine.best_params_}")
print(f"Fornisce accuracy in validation: {grid_fine.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid_fine.score(test_x, test_labels)
print(f"Risultato sul set indipendente: {accuracy_finale:.4f}")

La miglior configurazione: {'classify__activation_function': 'sigmoide', 'classify__epoche': 100000, 'classify__nn_learning_rate': 0.5, 'reduce_dim__n_components': 40, 'scaling': MinMaxScaler()}
Fornisce accuracy in validation: 0.9479
Risultato sul set indipendente: 0.9167


In [ ]:
import pandas as pd

results_df_fine = pd.DataFrame(grid_fine.cv_results_)
results_df_fine.to_pickle("results/results-nn-savgol-fine-GridSearch.pkl")
# Ciascuna combinazione di parametri è una riga
print(f"Numero totale di configurazioni provate: {results_df_fine.shape[0]}")

# Selezioniamo solo le colonne interessanti per pulire la vista
columns_to_show = [
    'param_scaling',
    'param_reduce_dim__n_components', 
    'param_classify__activation_function',
    'param_classify__epoche',
    'param_classify__nn_learning_rate', 
    'mean_test_score', 
    'std_test_score', 
    'mean_test_sensitivity',
    'std_test_sensitivity',
    'rank_test_score'
]

# Ordiniamo per classifica (rank_test_score)
analysis_fine = results_df_fine[columns_to_show].sort_values('rank_test_score')
renamed_fine = analysis_fine.rename(columns={
    'param_scaling': 'scaler',
    'param_reduce_dim__n_components': 'pca_n_components', 
    'param_classify__activation_function': 'activation_function',
    'param_classify__epoche': 'epoche',
    'param_classify__nn_learning_rate': 'learning_rate', 
    'mean_test_score': 'mean_score', 
    'std_test_score': 'std_score', 
    'mean_test_sensitivity': 'mean_sensitivity',
    'std_test_sensitivity': 'std_sensitivity',
    'rank_test_score': 'rank_score',
})

# Se si ha, è comodo aprire analysis in un viewer tipo Data Wrangler
renamed_fine.head(20)

Numero totale di configurazioni provate: 64


,scaler,pca_n_components,activation_function,epoche,learning_rate,mean_score,std_score,mean_sensitivity,std_sensitivity,rank_score
23,MinMaxScaler(),40.0,sigmoide,100000,0.9,0.947917,0.065716,0.995833,0.032005,1
27,MinMaxScaler(),35.0,sigmoide,500000,0.1,0.947917,0.065716,0.995833,0.032005,1
30,MinMaxScaler(),35.0,sigmoide,500000,0.5,0.947917,0.065716,0.995833,0.032005,1
20,MinMaxScaler(),40.0,sigmoide,100000,0.5,0.947917,0.065716,0.995833,0.032005,1
39,MinMaxScaler(),35.0,sigmoide,1000000,0.1,0.947917,0.065716,0.995833,0.032005,1
21,MinMaxScaler(),35.0,sigmoide,100000,0.9,0.947917,0.065716,0.995833,0.032005,1
41,MinMaxScaler(),40.0,sigmoide,1000000,0.1,0.947917,0.065716,0.995833,0.032005,1
43,MinMaxScaler(),37.0,sigmoide,1000000,0.5,0.947917,0.065716,0.995833,0.032005,1
40,MinMaxScaler(),37.0,sigmoide,1000000,0.1,0.947917,0.065716,0.995833,0.032005,1
6,MinMaxScaler(),35.0,sigmoide,20000,0.5,0.945833,0.066012,0.995833,0.032005,10
